In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df_agent = pd.read_csv("data/agent_roster.csv")
df_call = pd.read_csv("data/call_logs.csv")
df_summary = pd.read_csv("data/disposition_summary.csv")

# Understanding Agent Roaster Dataset

In [3]:
df_agent.head()

,agent_id,users_first_name,users_last_name,users_office_location,org_id
0,A001,AgentFirst1,AgentLast1,Bangalore,O1
1,A002,AgentFirst2,AgentLast2,Delhi,O1
2,A003,AgentFirst3,AgentLast3,Mumbai,O1
3,A004,AgentFirst4,AgentLast4,Bangalore,O3
4,A005,AgentFirst5,AgentLast5,Bangalore,O3


In [4]:
df_agent.shape

(20, 5)

In [5]:
df_agent.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   agent_id               20 non-null     object
 1   users_first_name       20 non-null     object
 2   users_last_name        20 non-null     object
 3   users_office_location  20 non-null     object
 4   org_id                 20 non-null     object
dtypes: object(5)
memory usage: 932.0+ bytes


In [6]:
df_agent["users_office_location"].unique()

array(['Bangalore', 'Delhi', 'Mumbai', 'Chennai'], dtype=object)

In [7]:
df_agent["org_id"].unique()

array(['O1', 'O3', 'O2'], dtype=object)

In [8]:
df_agent["agent_id"].nunique() # checking if there any duplicate entries. so we have 20 rows and 20 uniques so no duplicates.

20

- The agent_roster dataset contains information on 20 unique agents.
- These agents are distributed across 4 office locations: Bangalore, Delhi, Mumbai, and Chennai.
- They are employed under 3 different organizations, identified by unique org_id [o1, o2, o3] values.

# Understanding Call Logs Dataset

In [9]:
df_call.head()

,call_id,agent_id,org_id,installment_id,status,duration,created_ts,call_date
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28T15:40:00,2025-04-28
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28T02:41:00,2025-04-28
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28T19:42:00,2025-04-28
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28T09:52:00,2025-04-28
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28T12:58:00,2025-04-28


In [10]:
df_call.shape

(500, 8)

In [11]:
df_call.describe()

,duration
count,500.000000
mean,7.528660
std,4.450312
min,0.180000
25%,3.692500
50%,7.745000
75%,11.367500
max,14.900000


In [12]:
df_call.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   call_id         500 non-null    object 
 1   agent_id        500 non-null    object 
 2   org_id          500 non-null    object 
 3   installment_id  500 non-null    object 
 4   status          500 non-null    object 
 5   duration        500 non-null    float64
 6   created_ts      500 non-null    object 
 7   call_date       500 non-null    object 
dtypes: float64(1), object(7)
memory usage: 31.4+ KB


In [13]:
df_call["status"].unique()

array(['completed', 'no_answer', 'failed', 'connected'], dtype=object)

In [14]:
df_call["created_ts"].unique() # just to make sure all the dates are in same format

array(['2025-04-28T15:40:00', '2025-04-28T02:41:00',
       '2025-04-28T19:42:00', '2025-04-28T09:52:00',
       '2025-04-28T12:58:00', '2025-04-28T05:33:00',
       '2025-04-28T23:57:00', '2025-04-28T05:34:00',
       '2025-04-28T23:22:00', '2025-04-28T10:46:00',
       '2025-04-28T19:21:00', '2025-04-28T13:30:00',
       '2025-04-28T08:25:00', '2025-04-28T19:55:00',
       '2025-04-28T17:23:00', '2025-04-28T03:44:00',
       '2025-04-28T20:21:00', '2025-04-28T18:03:00',
       '2025-04-28T23:16:00', '2025-04-28T03:48:00',
       '2025-04-28T00:06:00', '2025-04-28T17:19:00',
       '2025-04-28T05:13:00', '2025-04-28T20:26:00',
       '2025-04-28T12:23:00', '2025-04-28T02:21:00',
       '2025-04-28T18:45:00', '2025-04-28T07:13:00',
       '2025-04-28T13:37:00', '2025-04-28T04:07:00',
       '2025-04-28T00:43:00', '2025-04-28T00:14:00',
       '2025-04-28T01:04:00', '2025-04-28T09:30:00',
       '2025-04-28T19:29:00', '2025-04-28T13:53:00',
       '2025-04-28T12:05:00', '2025-04-28T22:5

In [15]:
df_call["call_date"].unique()

array(['2025-04-28'], dtype=object)

In [16]:
df_call[df_call["duration"] == df_call["duration"].min()] # 'connected' calls under 20 seconds likely mean the customer asked 
                                                          # to be called later.

,call_id,agent_id,org_id,installment_id,status,duration,created_ts,call_date
72,C7658,A018,O1,L1997,connected,0.18,2025-04-28T08:04:00,2025-04-28
208,C2646,A004,O3,L1454,connected,0.18,2025-04-28T23:41:00,2025-04-28


In [17]:
df_call["call_id"].nunique() # probably have duplicates.

489

Believing this is the understanding of status

- `completed`: The call was successfully connected and finished properly — the agent and customer spoke, and the conversation ended normally.
- `connected`: The call was connected to the customer, but may not have been completed — could have been dropped, cut short, or transferred.
- `no_answer`: The customer's phone rang, but they didn't pick up within the allowed time — no conversation occurred.
- `failed`: The call did not go through at all — likely due to technical issues (e.g., invalid number, network failure, etc.).

Maybe we can scrape time from created_ts as we have date column

# Understanding Disposition Summary Dataset

In [18]:
df_summary.head()

,agent_id,org_id,call_date,login_time
0,A001,O1,2025-04-28,11:58
1,A002,O1,2025-04-28,10:05
2,A003,O1,2025-04-28,10:24
3,A004,O3,2025-04-28,8:13
4,A005,O3,2025-04-28,NaN


In [19]:
df_summary.shape

(20, 4)

In [20]:
df_summary.info() # we can see login_time as NAN lets handle it later

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   agent_id    20 non-null     object
 1   org_id      20 non-null     object
 2   call_date   20 non-null     object
 3   login_time  17 non-null     object
dtypes: object(4)
memory usage: 772.0+ bytes


In [21]:
 df_summary["call_date"].unique() # just to make sure all the dates are in same format

array(['2025-04-28'], dtype=object)

**Note**: Only thing we can do here is handle NAN

# Data Transformation

In [22]:
# 1. lets split time from created_ts column in CALL LOGS dataset using regex and drop created_ts

In [23]:
def scrape_time(timestamp):
    match = re.search(r'T(.*)', timestamp)
    if match:
        time_part = match.group(1)
        return time_part

df_call["time"] = df_call["created_ts"].apply(scrape_time)
df_call.drop(columns="created_ts", inplace=True)

In [24]:
df_call.head()

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28,02:41:00
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28,19:42:00
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28,09:52:00
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28,12:58:00


In [25]:
# 2. lets not delete the NAN in login_time in Disposition Summary because they maybe absent on #that day. 
# so lets create a column to check there present

In [26]:
df_summary["is_present"] = [1 if df_summary["login_time"].iloc[i] is not np.nan else 0 for i in range(len(df_summary))]

In [27]:
df_summary.head()

,agent_id,org_id,call_date,login_time,is_present
0,A001,O1,2025-04-28,11:58,1
1,A002,O1,2025-04-28,10:05,1
2,A003,O1,2025-04-28,10:24,1
3,A004,O3,2025-04-28,8:13,1
4,A005,O3,2025-04-28,NaN,0


In [28]:
# 3. lets handle duplicates because call_id should be unique as a primary key

In [29]:
df_call[df_call.duplicated(subset=['call_id'], keep=False)]

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00
15,C2489,A016,O1,L1773,completed,2.83,2025-04-28,03:44:00
19,C5371,A018,O1,L1787,failed,4.44,2025-04-28,03:48:00
28,C5342,A006,O1,L1540,no_answer,8.93,2025-04-28,07:13:00
44,C7658,A006,O1,L1497,no_answer,3.96,2025-04-28,07:17:00
72,C7658,A018,O1,L1997,connected,0.18,2025-04-28,08:04:00
75,C5371,A008,O1,L1163,completed,10.96,2025-04-28,13:03:00
78,C4164,A001,O1,L1408,failed,5.44,2025-04-28,09:30:00
101,C2988,A008,O1,L1467,connected,9.58,2025-04-28,15:51:00
142,C7658,A008,O1,L1500,connected,7.51,2025-04-28,23:34:00


In [30]:
df_call.drop_duplicates(subset=['call_id'], inplace=True)
df_call.reset_index()
df_call.shape

(489, 8)

# Joining tables

In [31]:
# joining call logs and agent roaster on agent_id

In [32]:
merged = df_call.merge(
    df_agent,
    on=['agent_id', 'org_id'],
    how='left',
    suffixes=('', '_agent')
)

In [33]:
final_merged = merged.merge(
    df_summary,
    on=['agent_id', 'org_id', 'call_date'],
    how='left',
    suffixes=('', '_disp')
)

In [34]:
final_merged

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time,users_first_name,users_last_name,users_office_location,login_time,is_present
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00,AgentFirst20,AgentLast20,Delhi,9:50,1
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28,02:41:00,AgentFirst18,AgentLast18,Bangalore,NaN,0
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28,19:42:00,AgentFirst18,AgentLast18,Bangalore,NaN,0
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28,09:52:00,AgentFirst7,AgentLast7,Bangalore,8:36,1
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28,12:58:00,AgentFirst3,AgentLast3,Mumbai,10:24,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,C5508,A010,O3,L1062,connected,5.36,2025-04-28,21:22:00,AgentFirst10,AgentLast10,Mumbai,11:11,1
485,C9120,A014,O3,L1048,failed,8.80,2025-04-28,21:57:00,AgentFirst14,AgentLast14,Mumbai,11:05,1
486,C9592,A013,O1,L1326,no_answer,1.26,2025-04-28,13:56:00,AgentFirst13,AgentLast13,Delhi,10:01,1
487,C5913,A005,O3,L1385,connected,14.62,2025-04-28,18:21:00,AgentFirst5,AgentLast5,Bangalore,NaN,0


In [35]:
final_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 489 entries, 0 to 488
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   call_id                489 non-null    object 
 1   agent_id               489 non-null    object 
 2   org_id                 489 non-null    object 
 3   installment_id         489 non-null    object 
 4   status                 489 non-null    object 
 5   duration               489 non-null    float64
 6   call_date              489 non-null    object 
 7   time                   489 non-null    object 
 8   users_first_name       489 non-null    object 
 9   users_last_name        489 non-null    object 
 10  users_office_location  489 non-null    object 
 11  login_time             407 non-null    object 
 12  is_present             489 non-null    int64  
dtypes: float64(1), int64(1), object(11)
memory usage: 49.8+ KB


In [36]:
# converting date to date format and time to time format
final_merged['call_date'] = pd.to_datetime(final_merged['call_date'])
final_merged['login_time'] = pd.to_datetime(final_merged['login_time'], format='%H:%M',).dt.time
final_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 489 entries, 0 to 488
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   call_id                489 non-null    object        
 1   agent_id               489 non-null    object        
 2   org_id                 489 non-null    object        
 3   installment_id         489 non-null    object        
 4   status                 489 non-null    object        
 5   duration               489 non-null    float64       
 6   call_date              489 non-null    datetime64[ns]
 7   time                   489 non-null    object        
 8   users_first_name       489 non-null    object        
 9   users_last_name        489 non-null    object        
 10  users_office_location  489 non-null    object        
 11  login_time             407 non-null    object        
 12  is_present             489 non-null    int64         
dtypes: da

# Feature Engineering

In [37]:
final_merged['total_call_made'] = (
    final_merged.groupby(['agent_id', 'call_date'])['call_id']
    .transform('count')
)

In [38]:
final_merged

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time,users_first_name,users_last_name,users_office_location,login_time,is_present,total_call_made
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00,AgentFirst20,AgentLast20,Delhi,09:50:00,1,21
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28,02:41:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28,19:42:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28,09:52:00,AgentFirst7,AgentLast7,Bangalore,08:36:00,1,24
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28,12:58:00,AgentFirst3,AgentLast3,Mumbai,10:24:00,1,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,C5508,A010,O3,L1062,connected,5.36,2025-04-28,21:22:00,AgentFirst10,AgentLast10,Mumbai,11:11:00,1,38
485,C9120,A014,O3,L1048,failed,8.80,2025-04-28,21:57:00,AgentFirst14,AgentLast14,Mumbai,11:05:00,1,29
486,C9592,A013,O1,L1326,no_answer,1.26,2025-04-28,13:56:00,AgentFirst13,AgentLast13,Delhi,10:01:00,1,22
487,C5913,A005,O3,L1385,connected,14.62,2025-04-28,18:21:00,AgentFirst5,AgentLast5,Bangalore,NaT,0,28


In [39]:
final_merged['unique_loan'] = (
    final_merged.groupby(['agent_id', 'call_date'])['installment_id']
    .transform(lambda x: x.nunique())
)

In [40]:
final_merged

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time,users_first_name,users_last_name,users_office_location,login_time,is_present,total_call_made,unique_loan
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00,AgentFirst20,AgentLast20,Delhi,09:50:00,1,21,21
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28,02:41:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28,19:42:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28,09:52:00,AgentFirst7,AgentLast7,Bangalore,08:36:00,1,24,23
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28,12:58:00,AgentFirst3,AgentLast3,Mumbai,10:24:00,1,21,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,C5508,A010,O3,L1062,connected,5.36,2025-04-28,21:22:00,AgentFirst10,AgentLast10,Mumbai,11:11:00,1,38,38
485,C9120,A014,O3,L1048,failed,8.80,2025-04-28,21:57:00,AgentFirst14,AgentLast14,Mumbai,11:05:00,1,29,29
486,C9592,A013,O1,L1326,no_answer,1.26,2025-04-28,13:56:00,AgentFirst13,AgentLast13,Delhi,10:01:00,1,22,22
487,C5913,A005,O3,L1385,connected,14.62,2025-04-28,18:21:00,AgentFirst5,AgentLast5,Bangalore,NaT,0,28,27


In [41]:
final_merged['complete_call_count'] = (
    final_merged[final_merged['status'] == 'completed']
    .groupby(['agent_id', 'call_date'])["status"]
    .transform('count')
)

In [42]:
final_merged

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time,users_first_name,users_last_name,users_office_location,login_time,is_present,total_call_made,unique_loan,complete_call_count
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00,AgentFirst20,AgentLast20,Delhi,09:50:00,1,21,21,7.0
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28,02:41:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33,NaN
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28,19:42:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33,NaN
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28,09:52:00,AgentFirst7,AgentLast7,Bangalore,08:36:00,1,24,23,NaN
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28,12:58:00,AgentFirst3,AgentLast3,Mumbai,10:24:00,1,21,21,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,C5508,A010,O3,L1062,connected,5.36,2025-04-28,21:22:00,AgentFirst10,AgentLast10,Mumbai,11:11:00,1,38,38,NaN
485,C9120,A014,O3,L1048,failed,8.80,2025-04-28,21:57:00,AgentFirst14,AgentLast14,Mumbai,11:05:00,1,29,29,NaN
486,C9592,A013,O1,L1326,no_answer,1.26,2025-04-28,13:56:00,AgentFirst13,AgentLast13,Delhi,10:01:00,1,22,22,NaN
487,C5913,A005,O3,L1385,connected,14.62,2025-04-28,18:21:00,AgentFirst5,AgentLast5,Bangalore,NaT,0,28,27,NaN


In [43]:
final_merged['complete_rate'] = final_merged.apply(
    lambda row: (row['complete_call_count'] / row['total_call_made'] 
                if row['complete_call_count'] > 0 
                else 0),
    axis=1
)

In [44]:
final_merged

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time,users_first_name,users_last_name,users_office_location,login_time,is_present,total_call_made,unique_loan,complete_call_count,complete_rate
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00,AgentFirst20,AgentLast20,Delhi,09:50:00,1,21,21,7.0,0.333333
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28,02:41:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33,NaN,0.000000
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28,19:42:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33,NaN,0.000000
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28,09:52:00,AgentFirst7,AgentLast7,Bangalore,08:36:00,1,24,23,NaN,0.000000
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28,12:58:00,AgentFirst3,AgentLast3,Mumbai,10:24:00,1,21,21,8.0,0.380952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,C5508,A010,O3,L1062,connected,5.36,2025-04-28,21:22:00,AgentFirst10,AgentLast10,Mumbai,11:11:00,1,38,38,NaN,0.000000
485,C9120,A014,O3,L1048,failed,8.80,2025-04-28,21:57:00,AgentFirst14,AgentLast14,Mumbai,11:05:00,1,29,29,NaN,0.000000
486,C9592,A013,O1,L1326,no_answer,1.26,2025-04-28,13:56:00,AgentFirst13,AgentLast13,Delhi,10:01:00,1,22,22,NaN,0.000000
487,C5913,A005,O3,L1385,connected,14.62,2025-04-28,18:21:00,AgentFirst5,AgentLast5,Bangalore,NaT,0,28,27,NaN,0.000000


In [45]:
final_merged['average_call'] = (
    final_merged.groupby(['agent_id', 'call_date'])['duration']
    .transform('mean')
)

In [46]:
final_merged['average_call'] = final_merged['average_call'].round(1)

In [47]:
final_merged

,call_id,agent_id,org_id,installment_id,status,duration,call_date,time,users_first_name,users_last_name,users_office_location,login_time,is_present,total_call_made,unique_loan,complete_call_count,complete_rate,average_call
0,C5333,A020,O2,L1826,completed,5.68,2025-04-28,15:40:00,AgentFirst20,AgentLast20,Delhi,09:50:00,1,21,21,7.0,0.333333,6.6
1,C3045,A018,O1,L1996,no_answer,14.27,2025-04-28,02:41:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33,NaN,0.000000,7.7
2,C5803,A018,O1,L1849,failed,11.01,2025-04-28,19:42:00,AgentFirst18,AgentLast18,Bangalore,NaT,0,34,33,NaN,0.000000,7.7
3,C2139,A007,O1,L1046,connected,9.02,2025-04-28,09:52:00,AgentFirst7,AgentLast7,Bangalore,08:36:00,1,24,23,NaN,0.000000,7.0
4,C4814,A003,O1,L1887,completed,2.42,2025-04-28,12:58:00,AgentFirst3,AgentLast3,Mumbai,10:24:00,1,21,21,8.0,0.380952,7.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,C5508,A010,O3,L1062,connected,5.36,2025-04-28,21:22:00,AgentFirst10,AgentLast10,Mumbai,11:11:00,1,38,38,NaN,0.000000,8.1
485,C9120,A014,O3,L1048,failed,8.80,2025-04-28,21:57:00,AgentFirst14,AgentLast14,Mumbai,11:05:00,1,29,29,NaN,0.000000,7.5
486,C9592,A013,O1,L1326,no_answer,1.26,2025-04-28,13:56:00,AgentFirst13,AgentLast13,Delhi,10:01:00,1,22,22,NaN,0.000000,6.9
487,C5913,A005,O3,L1385,connected,14.62,2025-04-28,18:21:00,AgentFirst5,AgentLast5,Bangalore,NaT,0,28,27,NaN,0.000000,7.2


In [66]:
def top_performer(df):
    df.sort_values(by=["complete_rate", "unique_loan"], ascending=False, inplace = True)
    return f"{df.iloc[0].users_first_name +" " +df.iloc[0].users_last_name} ({round(df.iloc[0].complete_rate *100,0)}% connect rate)"

In [74]:
for date in final_merged["call_date"].unique():
    print("Summary for", date)
    date_wise = final_merged[final_merged["call_date"] == date]
    print("Top Performers: ", top_performer(date_wise))
    print("Total Active Agents: ", date_wise[date_wise['is_present'] == True]['agent_id'].nunique())
    print("Average Duration: ", date_wise['average_call'].mean())

Summary for 2025-04-28 00:00:00
Top Performers:  AgentFirst3 AgentLast3 (38.0% connect rate)
Total Active Agents:  17
Average Duration:  7.54601226993865


In [77]:
df_agent.columns()

TypeError: 'Index' object is not callable